# Lorenz-96 data assimilation: five observation configurations with localized weighted ensemble-Jacobian likelihood scaling

This notebook keeps the five observation configurations and the parameter-sweep workflow from the consolidated Lorenz-96 notebook, but replaces the global $K$-dimensional weighted regression by **many small periodic local regressions**.

The five observation configurations are:

1. `random`: fixed linear observations plus random nonlinear observations.
2. `timevarying`: nonlinear coverage changes during the DA window.
3. `nooverlap`: nonlinear observations avoid recently observed locations.
4. `block`: periodic random blocks.
5. `stencil`: periodic structured/stencil observations.

The observation geometry, observation-noise model, optional diffusion-dependent denominator, latent inflation, posterior spread adjustment, deterministic sweep comparison, and fine-tuning interface are retained.

## Local weighted-$F$ regression

The EDM network quantity is recovered from the denoiser by

$$
U_e = c_{\mathrm{in}}X_e,
\qquad
F_e = \frac{x_{0,e}-c_{\mathrm{skip}}X_e}{c_{\mathrm{out}}}.
$$

Instead of estimating one global $K\times K$ Jacobian, for each observed state index $j$ the code uses the periodic neighborhood

$$
\mathcal N_j = \{j-R,\ldots,j,\ldots,j+R\}\pmod K
$$

and fits

$$
F_{e,j}-\bar F_j
\approx
b_j^T\left(U_{e,\mathcal N_j}-\bar U_{\mathcal N_j}\right).
$$

Thus the local regression dimension is

$$
d_{\mathrm{loc}}=2R+1,
$$

rather than $K=1000$. With the default `jf_local_radius=2`, $d_{\mathrm{loc}}=5$, which is well below `NE=36`.

The local transpose action is assembled by accumulating the overlapping local contributions,

$$
(J_F^T g)_k
\approx
\sum_{j:\,k\in\mathcal N_j} b_{j,k}\,g_j.
$$

Gaussian ensemble weights are computed separately for each local regression, and ridge regularization is retained.

## Four guidance/Jacobian scales

Let $g_{\mathrm{lin}}$ and $g_{\mathrm{nonlin}}$ denote the likelihood-gradient contributions from the linear and nonlinear observation sets. The two existing guidance scales are kept as the $\Lambda_L$ factors,

$$
\widetilde g_{\mathrm{lin}}=\lambda_{\mathrm{lin}}g_{\mathrm{lin}},
\qquad
\widetilde g_{\mathrm{nonlin}}=\lambda_{\mathrm{nonlin}}g_{\mathrm{nonlin}}.
$$

Two additional scalar $J_F$ factors are added, analogous to the componentwise $\alpha_F$ scaling in the Lorenz-63 version:

$$
A_e
=
c_{\mathrm{skip}}
\left(
\widetilde g_{\mathrm{lin},e}
+
\widetilde g_{\mathrm{nonlin},e}
\right)
+
c_{\mathrm{out}}c_{\mathrm{in}}
\left[
\alpha_{F,\mathrm{lin}}J_F^T\widetilde g_{\mathrm{lin},e}
+
\alpha_{F,\mathrm{nonlin}}J_F^T\widetilde g_{\mathrm{nonlin},e}
\right].
$$

In code these four tunable parameters are:

- `guidance_scale_lin`
- `guidance_scale_nonlin`
- `jf_scale_lin`
- `jf_scale_nonlin`

The relaxed residual normalization is retained:

$$
c
=
\alpha_{\mathrm{soft}}
\sqrt{
n_{\mathrm{lin}}\sigma_{\mathrm{lin}}^2
+
n_{\mathrm{nonlin}}\sigma_{\mathrm{nonlin}}^2
},
\qquad
\rho_e
=
\sqrt{\|r_e\|_2^2+c^2}.
$$

The reverse-diffusion correction is evaluated at both Heun stages.

## Fine-tuning

The parameter-sweep section can override only the quantities being tuned. Useful parameters include:

- `guidance_scale_lin`
- `guidance_scale_nonlin`
- `jf_scale_lin`
- `jf_scale_nonlin`
- `jf_local_radius`
- `residual_soften_alpha`
- `jf_weight_bandwidth_factor`
- `intermediate_guidance_multiplier`
- `latent_inflation`
- `posterior_gamma`

**Important:** re-tune the four guidance/Jacobian scales after changing the regression from global to local.


In [2]:
import os
import pickle
import time

import dnnlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import tqdm
from tqdm.autonotebook import tqdm as notebook_tqdm

from project_utils.normalization import (
    denormalize_tensor,
    load_norm_stats,
    normalize_tensor,
    resolve_norm_stats_path,
)
from project_utils.paths import PROJECT_ROOT, dataset_trajectory_dir, CKPT_DIR
from project_utils.yaml_simple import load_simple_yaml
from turbulence.lorenz96 import Lorenz96Params, Lorenz96Solver


### Utilities

**Shape convention:** `C=1`, sequence length `K`, stored as `(1, K, 1)`.


In [3]:
class StackedRandomGenerator:
    def __init__(self, device, seeds):
        self.generators = [
            torch.Generator(device).manual_seed(int(s) % (1 << 32)) for s in seeds
        ]

    def randn(self, size, **kwargs):
        if len(size) < 1 or len(self.generators) != size[0]:
            raise ValueError(
                f"size[0]={size[0] if size else None} must match n_generators={len(self.generators)}"
            )
        return torch.stack(
            [torch.randn(size[1:], generator=g, **kwargs) for g in self.generators]
        )

    def randn_like(self, input):
        return self.randn(
            input.shape, dtype=input.dtype, layout=input.layout, device=input.device
        )

def frame_tensor(x: torch.Tensor, k: int) -> torch.Tensor:
    return x.reshape(1, k, 1)

def make_initial_condition(k: int, F: float, *, device, dtype: torch.dtype) -> torch.Tensor:
    x = torch.full((k,), float(F), device=device, dtype=dtype)
    x[0] = x[0] + torch.tensor(0.01, device=device, dtype=dtype)
    return x


def build_condition_from_history(frames_norm, history):
    # frames_norm: list of (C, H, W) or (T, C, H, W), already normalized -> (1, history*C, H, W)
    if isinstance(frames_norm, torch.Tensor):
        block = frames_norm[-history:]  # (history, C, H, W)
    else:
        block = torch.stack(frames_norm[-history:], dim=0)
    C, H, W = block.shape[1], block.shape[2], block.shape[3]
    return block.reshape(history * C, H, W).unsqueeze(0)

def build_condition_from_history_batch(histories, history):
    return torch.cat(
        [build_condition_from_history(h, history) for h in histories],
        dim=0,
    )

def perturb_state(frame, std, seed):
    g = torch.Generator().manual_seed(int(seed) % (1 << 32))
    return frame + std * torch.randn(frame.shape, generator=g, dtype=frame.dtype, device=frame.device)


def ensemble_last_frames(histories):
    return torch.stack([h[-1] for h in histories], dim=0)

def ensemble_mean_rmse(histories, x_truth):
    ens = ensemble_last_frames(histories)
    mean = ens.mean(dim=0)
    spread = ens.std(dim=0, correction=0).mean().item() if ens.shape[0] > 1 else 0.0
    rmse = torch.sqrt(torch.mean((mean - x_truth) ** 2)).item()
    return rmse, spread


def to_seq(traj: torch.Tensor) -> torch.Tensor:
    return traj[:, 0, :, 0]


In [4]:
# --- User configuration ---
L96_CONFIG = os.path.join(PROJECT_ROOT, "configs/turbulence/lorenz96_default.yaml")
DATA_ROOT = "LORENZ96"
TRAJECTORY_SUBDIR = "full_path"

NETWORK_PKL = os.path.join(
    CKPT_DIR, "l96_dim_1000_dt_01.pkl",
)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEED = 0
SEED_DM = 0

K = 1000
T_BURN_IN = 2000
N_DA_CYCLES = 300
OUTPUT_INTERVAL = 0.01

HISTORY = 1

NE = 36
ENSEMBLE_PERTURB_STD = 0.05

device = torch.device(DEVICE)
trajectory_dir = dataset_trajectory_dir(DATA_ROOT, TRAJECTORY_SUBDIR)
norm_stats_path = resolve_norm_stats_path(trajectory_dir, None)

## Phase 1: True L96 burn-in

In [5]:
cfg = load_simple_yaml(L96_CONFIG)
l96_cfg = dict(cfg.get("lorenz96", {}))
l96_cfg["device"] = str(device)

params = Lorenz96Params(K=K, **l96_cfg)
params.validate()
timesteps_per_output = max(1, int(OUTPUT_INTERVAL / params.dt))
dtype = torch.float32 if params.precision == "single" else torch.float64

x0 = make_initial_condition(K, params.F, device=device, dtype=dtype)
solver = Lorenz96Solver(x0, **params.to_core_kwargs())

burn_frames = [frame_tensor(solver.fields()["x"].detach().cpu(), K)]
for _ in tqdm.tqdm(range(T_BURN_IN), desc="L96 burn-in"):
    solver.step(timesteps_per_output)
    burn_frames.append(frame_tensor(solver.fields()["x"].detach().cpu(), K))

state_after_burn = {k: v.clone() for k, v in solver.get_state().items()}
path_burn = torch.stack(burn_frames, dim=0)
print("Burn-in frames:", tuple(path_burn.shape))


L96 burn-in: 100%|██████████| 2000/2000 [00:00<00:00, 3354.20it/s]


Burn-in frames: (2001, 1, 1000, 1)


## Phase 2: Load trained conditional diffusion model + norm stats

In [6]:
if not os.path.isfile(NETWORK_PKL):
    raise FileNotFoundError(
        f"Set NETWORK_PKL. Not found: {NETWORK_PKL}\n"
        "Train with: torchrun scripts/train/train_cond.py runs=lorenz96_path_next"
    )

norm_stats = load_norm_stats(norm_stats_path)
with dnnlib.util.open_url(NETWORK_PKL, verbose=True) as f:
    net = pickle.load(f)["ema"].to(device).eval()

n_target = 1
n_cond = HISTORY * n_target
assert net.img_input_channels == n_target
assert net.img_cond_channels == n_cond
assert net.img_res_h == K and net.img_res_w == 1

print(f"Loaded {NETWORK_PKL}")
print(f"Model: {net.model_name}")

Loaded /home/toanhuynh/GitHub/glocal-condiff-turb/checkpoints/l96_dim_1000_dt_01.pkl
Model: l96_unet1d_tiny


### Observation operator (sparse direct state)

Likelihood guidance is applied inside `edm_sampler_obs` when `obs["y_obs"]` is set.


In [7]:
def flat_index(i, K, channel=0):
    return int(channel) * int(K) + int(i)


def random_obs_flat_indices(K, obs_count, obs_channels, seed=0, device="cpu"):
    rng = np.random.default_rng(seed)
    all_idx = []
    for c in obs_channels:
        base = int(c) * int(K)
        all_idx.extend(range(base, base + int(K)))
    flat_idx = rng.choice(np.array(all_idx, dtype=np.int64), size=obs_count, replace=False)
    return torch.tensor(flat_idx, dtype=torch.long, device=device)


def obs_mask_vector(flat_idx, total_size, device=None, dtype=torch.float32):
    dev = device if device is not None else flat_idx.device
    m = torch.zeros(total_size, device=dev, dtype=dtype)
    m[flat_idx.to(dev)] = 1.0
    return m


def observe_state(x_phys, flat_idx, obs_sigma, generator=None):
    flat = x_phys.reshape(-1)
    idx = flat_idx.to(flat.device)
    y = flat[idx].to(torch.float32)
    if obs_sigma > 0:
        y = y + obs_sigma * torch.randn(y.shape, device=y.device, generator=generator)
    return y


def observation_rmse(x_phys, y_obs, flat_idx):
    flat = x_phys.reshape(-1)
    idx = flat_idx.cpu()
    return torch.sqrt(torch.mean((flat[idx] - y_obs.reshape(-1).cpu()) ** 2)).item()

### EDM sampling with weighted ensemble-Jacobian likelihood scaling

The sampler below preserves the existing observation likelihood and optional denominator. The change is the likelihood transport: the raw denoised-state likelihood gradient is scaled by the linear/nonlinear $\Lambda_L$ factors, transformed by the weighted ensemble estimate of $J_F^T$, normalized by the softened residual norm, and inserted at both Heun stages.

The weighted regression is centered at $\bar U$ and uses

$$
h = \beta_h\,\operatorname{median}_e\|U_e-\bar U\|_2.
$$

For high-dimensional Lorenz-96 states, the helper computes the ridge-regression action in ensemble space rather than explicitly forming $J_F$.


In [8]:
NUM_STEPS = 30

In [9]:
def phys_scale_at_flat_idx(flat_idx, norm_stats, seq_len, n_channels, device, dtype=torch.float64):
    flat_idx = flat_idx.to(device)
    block = int(seq_len)
    channels = flat_idx // block
    if (channels < 0).any() or (channels >= n_channels).any():
        raise ValueError(f"flat_idx out of range for (C={n_channels}, K={seq_len}) channel-major layout")
    if norm_stats.get("scope", "global") == "global":
        s = 0.5 * float(norm_stats["vmax"] - norm_stats["vmin"])
        return torch.full((flat_idx.numel(),), s, device=device, dtype=dtype)
    ch_scale = 0.5 * (norm_stats["vmax_per_channel"] - norm_stats["vmin_per_channel"]).to(device=device, dtype=dtype)
    return ch_scale[channels]


def check_obs_guidance_setup(norm_stats, flat_idx, n_channels, seq_len, device="cpu"):
    if norm_stats.get("scope", "global") == "global":
        lo = float(norm_stats["vmin"])
        hi = float(norm_stats["vmax"])
        x = torch.rand(n_channels, seq_len, 1, device=device) * (hi - lo) + lo
    else:
        vmin_c = norm_stats["vmin_per_channel"].to(device)
        vmax_c = norm_stats["vmax_per_channel"].to(device)
        x = torch.rand(n_channels, seq_len, 1, device=device)
        for c in range(n_channels):
            x[c] = x[c] * float(vmax_c[c] - vmin_c[c]) + float(vmin_c[c])

    err = (denormalize_tensor(normalize_tensor(x, norm_stats), norm_stats) - x).abs().max()
    if float(err) > 1e-4:
        raise RuntimeError(f"norm/denorm round-trip failed: max err {float(err):.2e}")

    idx = flat_idx.to(device)
    if idx.numel() == 0:
        return

    scales = phys_scale_at_flat_idx(idx, norm_stats, seq_len, n_channels, device)
    for k in (0, idx.numel() // 2, idx.numel() - 1):
        ch = int((idx[k] // seq_len).item())
        if norm_stats.get("scope", "global") == "global":
            ref = 0.5 * float(norm_stats["vmax"] - norm_stats["vmin"])
        else:
            ref = float(0.5 * (norm_stats["vmax_per_channel"][ch] - norm_stats["vmin_per_channel"][ch]))
        if abs(float(scales[k]) - ref) > 1e-5:
            raise RuntimeError(f"phys_scale mismatch at flat_idx={int(idx[k])}: {scales[k]} vs {ref}")

    print(f"obs guidance checks OK (round-trip err={float(err):.2e})")


def edm_rt2(sigma, sigma_data):
    s2 = sigma * sigma
    sd2 = sigma_data * sigma_data
    return s2 * sd2 / (s2 + sd2)


def obs_guidance_denom(sigma, obs_sigma, sigma_data, ell):
    return obs_sigma**2 + edm_rt2(sigma, sigma_data) * (ell * ell)


def obs_guidance_h(sigma, obs_sigma, sigma_data, ell):
    return obs_sigma**2 / obs_guidance_denom(sigma, obs_sigma, sigma_data, ell)


def _edm_sigma_data(net, fallback=0.5):
    value = getattr(net, "sigma_data", fallback)
    try:
        return float(torch.as_tensor(value).detach().cpu().reshape(-1)[0].item())
    except Exception:
        return float(value)


def _c_skip(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sd.pow(2) / (sigma.pow(2) + sd.pow(2))


def _c_in(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return 1.0 / torch.sqrt(sigma.pow(2) + sd.pow(2))


def _c_out(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sigma * sd / torch.sqrt(sigma.pow(2) + sd.pow(2))


def _c_out_c_in(t_stage, sigma_data, device, dtype):
    sigma = torch.as_tensor(t_stage, device=device, dtype=dtype)
    sd = torch.as_tensor(sigma_data, device=device, dtype=dtype)
    return sigma * sd / (sigma.pow(2) + sd.pow(2))


def _local_weighted_fbar_jf_transpose_actions(
    x_noisy, x0_denoised, g_list, t_stage, sigma_data, center_idx,
    radius=2, bandwidth_factor=1.0, ridge_rel=1e-6, ridge_abs=1e-10, eps=1e-12
):
    """Compute localized J_F^T g actions for periodic 1D Lorenz-96 neighborhoods.

    For each observed output index j, fit
        F_j - Fbar_j ~= b_j^T (U_N(j) - Ubar_N(j)),
    with N(j) = {j-radius, ..., j+radius} modulo K.

    The fitted b_j is the local row of J_F.  J_F^T g is assembled by
    accumulating g_j b_j back onto the overlapping local input neighborhoods.

    The regression is solved independently for every observed center, with
    Gaussian ensemble weights and ridge regularization.  The same fitted local
    regressions are reused for every guidance component supplied in g_list.
    """
    device, dtype = x_noisy.device, x_noisy.dtype
    if x_noisy.ndim != 4 or x_noisy.shape[1] != 1 or x_noisy.shape[-1] != 1:
        raise ValueError("Localized L96 regression expects shape (Ne, 1, K, 1).")

    ne, k = x_noisy.shape[0], x_noisy.shape[-2]
    radius = int(radius)
    if radius < 0:
        raise ValueError("jf_local_radius must be nonnegative.")

    centers = torch.unique(center_idx.to(device=device, dtype=torch.long), sorted=True)
    if centers.numel() == 0:
        return [torch.zeros_like(g) for g in g_list]

    offsets = torch.arange(-radius, radius + 1, device=device, dtype=torch.long)
    patch_idx = (centers[:, None] + offsets[None, :]) % k
    local_dim = patch_idx.shape[1]

    cskip = _c_skip(t_stage, sigma_data, device, dtype)
    cin = _c_in(t_stage, sigma_data, device, dtype)
    cout = _c_out(t_stage, sigma_data, device, dtype)

    u = (cin * x_noisy).reshape(ne, k)
    f = ((x0_denoised - cskip * x_noisy) / torch.clamp(cout, min=eps)).reshape(ne, k)

    # Shapes: local U = (n_centers, Ne, local_dim), center F = (n_centers, Ne).
    u_local = u[:, patch_idx].permute(1, 0, 2).contiguous()
    f_center = f[:, centers].transpose(0, 1).contiguous()

    du = u_local - u_local.mean(dim=1, keepdim=True)
    df = f_center - f_center.mean(dim=1, keepdim=True)

    distances = torch.linalg.vector_norm(du, ord=2, dim=2)
    median_distance = torch.median(distances, dim=1).values
    rms_distance = torch.sqrt(torch.mean(distances.pow(2), dim=1)).clamp_min(eps)
    median_distance = torch.where(median_distance > eps, median_distance, rms_distance)

    h = (torch.as_tensor(float(bandwidth_factor), device=device, dtype=dtype) * median_distance).clamp_min(eps)
    weights = torch.exp(-0.5 * (distances / h[:, None]).pow(2))
    weights = weights / weights.mean(dim=1, keepdim=True).clamp_min(eps)
    sqrt_w = torch.sqrt(torch.clamp(weights, min=0.0))

    a = sqrt_w[:, :, None] * du
    c = sqrt_w * df

    gram = torch.einsum("snp,snq->spq", a, a)
    rhs = torch.einsum("snp,sn->sp", a, c)

    gram_scale = gram.diagonal(dim1=-2, dim2=-1).abs().sum(dim=1) / max(local_dim, 1)
    ridge = (
        torch.as_tensor(float(ridge_abs), device=device, dtype=dtype)
        + torch.as_tensor(float(ridge_rel), device=device, dtype=dtype) * gram_scale
    )
    system = gram + ridge[:, None, None] * torch.eye(local_dim, device=device, dtype=dtype).unsqueeze(0)

    try:
        b = torch.linalg.solve(system, rhs.unsqueeze(-1)).squeeze(-1)
    except RuntimeError:
        b = (torch.linalg.pinv(system) @ rhs.unsqueeze(-1)).squeeze(-1)
    if not torch.isfinite(b).all():
        b = (torch.linalg.pinv(system) @ rhs.unsqueeze(-1)).squeeze(-1)

    # Accumulate overlapping local transpose contributions.
    scatter_idx = patch_idx.reshape(1, -1).expand(ne, -1)
    outputs = []
    for g in g_list:
        gf = g.reshape(ne, k)
        g_center = gf[:, centers].transpose(0, 1).contiguous()
        contribution = (g_center[:, :, None] * b[:, None, :]).permute(1, 0, 2).reshape(ne, -1)

        out = torch.zeros((ne, k), device=device, dtype=dtype)
        out.scatter_add_(1, scatter_idx, contribution)
        outputs.append(out.reshape_as(g))

    return outputs


def edm_sampler_obs(
    net, latents, conditions, obs=None, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float("inf"), S_noise=1
):
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    use_obs = obs is not None and obs.get("y_obs") is not None

    if use_obs:
        device = latents.device
        obs_sigma_lin = float(obs["obs_sigma_lin"])
        obs_sigma_nonlin = float(obs["obs_sigma_nonlin"])
        guidance_scale_lin = float(obs.get("guidance_scale_lin", 1.0))
        guidance_scale_nonlin = float(obs.get("guidance_scale_nonlin", 1.0))
        jf_scale_lin = float(obs.get("jf_scale_lin", 1.0))
        jf_scale_nonlin = float(obs.get("jf_scale_nonlin", 1.0))
        jf_local_radius = int(obs.get("jf_local_radius", 2))
        sigma_data = float(obs.get("sigma_data", 1.0))
        phi_nonlin = obs.get("phi_nonlin", lambda u: u)
        ell_lin = float(obs.get("obs_ell_lin", 1.0))
        ell_nonlin = float(obs.get("obs_ell_nonlin", 1.0))
        use_denom = bool(obs.get("use_denom", False))
        norm_stats = obs["norm_stats"]

        idx_lin = obs["idx_lin"].to(device)
        idx_nonlin = obs["idx_nonlin"].to(device)
        idx_all = torch.unique(torch.cat([idx_lin, idx_nonlin]), sorted=True)
        y_obs = obs["y_obs"].to(device, torch.float64)
        if y_obs.dim() == 1:
            y_obs = y_obs.unsqueeze(0)

        residual_soften_alpha = float(obs.get("residual_soften_alpha", 1.0))
        jf_weight_bandwidth_factor = float(obs.get("jf_weight_bandwidth_factor", 1.0))
        jf_ridge_rel = float(obs.get("jf_ridge_rel", 1e-6))
        jf_ridge_abs = float(obs.get("jf_ridge_abs", 1e-10))
        intermediate_guidance_multiplier = float(obs.get("intermediate_guidance_multiplier", 1.0))
        edm_sigma_data = _edm_sigma_data(net, fallback=float(obs.get("edm_sigma_data_fallback", 0.5)))

        n_lin = idx_lin.numel()
        n_nonlin = idx_nonlin.numel()
        residual_soften_c = residual_soften_alpha * np.sqrt(
            n_lin * obs_sigma_lin**2 + n_nonlin * obs_sigma_nonlin**2
        )
        residual_soften_c = max(float(residual_soften_c), 1e-12)

    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])])

    all_abs_dt = torch.abs(t_steps[1:] - t_steps[:-1])
    mean_abs_dt = all_abs_dt.mean().clamp_min(1e-12)

    x_next = latents.to(torch.float64) * t_steps[0]
    neg_log_likelihoods = [] if use_obs else None

    def likelihood_stage(x0_norm, sigma_for_denom):
        x0_leaf = x0_norm.detach().requires_grad_(True)
        x0_flat = x0_leaf.reshape(x0_leaf.shape[0], -1)

        gx_lin = x0_flat[:, idx_lin]
        gx_nonlin = phi_nonlin(x0_flat[:, idx_nonlin])
        gx = torch.cat([gx_lin, gx_nonlin], dim=1)
        resid = gx - y_obs

        if use_denom:
            sigma = float(torch.as_tensor(sigma_for_denom).reshape(-1)[0].item())
            sigma_vec = torch.cat([
                torch.full((n_lin,), obs_sigma_lin, device=device, dtype=torch.float64),
                torch.full((n_nonlin,), obs_sigma_nonlin, device=device, dtype=torch.float64),
            ])
            ell_vec = torch.cat([
                torch.full((n_lin,), ell_lin, device=device, dtype=torch.float64),
                torch.full((n_nonlin,), ell_nonlin, device=device, dtype=torch.float64),
            ])
            denom = obs_guidance_denom(sigma, sigma_vec, sigma_data, ell_vec)
        else:
            denom = torch.cat([
                torch.full((n_lin,), obs_sigma_lin**2, device=device, dtype=torch.float64),
                torch.full((n_nonlin,), obs_sigma_nonlin**2, device=device, dtype=torch.float64),
            ])

        nll = 0.5 * (resid.pow(2) / denom).sum(dim=1)
        raw_guidance = torch.autograd.grad(-nll.sum(), x0_leaf)[0].detach()

        # Split the raw likelihood gradient by observation type.
        raw_flat = raw_guidance.reshape(raw_guidance.shape[0], -1)
        g_lin_flat = torch.zeros_like(raw_flat)
        g_nonlin_flat = torch.zeros_like(raw_flat)
        if n_lin:
            g_lin_flat[:, idx_lin] = raw_flat[:, idx_lin]
        if n_nonlin:
            g_nonlin_flat[:, idx_nonlin] = raw_flat[:, idx_nonlin]

        # Lambda_L factors apply to BOTH the direct and J_F branches.
        g_lin = (guidance_scale_lin * g_lin_flat).reshape_as(raw_guidance)
        g_nonlin = (guidance_scale_nonlin * g_nonlin_flat).reshape_as(raw_guidance)

        resid_norm = torch.linalg.vector_norm(resid.detach(), ord=2, dim=1)
        soft_resid_norm = torch.sqrt(resid_norm.pow(2) + residual_soften_c**2)
        return g_lin, g_nonlin, soft_resid_norm, nll.detach()

    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat**2 - t_cur**2).sqrt() * S_noise * randn_like(x_cur)

        dt = t_next - t_hat
        abs_dt = torch.abs(dt).clamp_min(1e-12)
        adaptive_dt_factor = mean_abs_dt / abs_dt

        with torch.no_grad():
            x0_cur = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
        d_cur = (x_hat - x0_cur) / t_hat

        if use_obs:
            g_lin_cur, g_nonlin_cur, soft_resid_norm_cur, nll_cur = likelihood_stage(x0_cur, t_hat)
            cskip_cur = _c_skip(t_hat, edm_sigma_data, x_hat.device, x_hat.dtype)
            coutcin_cur = _c_out_c_in(t_hat, edm_sigma_data, x_hat.device, x_hat.dtype)

            jf_lin_cur, jf_nonlin_cur = _local_weighted_fbar_jf_transpose_actions(
                x_hat, x0_cur, [g_lin_cur, g_nonlin_cur], t_hat, edm_sigma_data, idx_all,
                radius=jf_local_radius, bandwidth_factor=jf_weight_bandwidth_factor,
                ridge_rel=jf_ridge_rel, ridge_abs=jf_ridge_abs
            )

            direct_cur = g_lin_cur + g_nonlin_cur
            jf_cur = jf_scale_lin * jf_lin_cur + jf_scale_nonlin * jf_nonlin_cur
            jac_action_cur = cskip_cur * direct_cur + coutcin_cur * jf_cur

            scale_shape = [soft_resid_norm_cur.shape[0]] + [1] * (direct_cur.ndim - 1)
            scale_cur = adaptive_dt_factor / soft_resid_norm_cur.reshape(scale_shape)

        x_euler = x_hat + dt * d_cur
        if use_obs:
            x_euler = x_euler - dt * t_hat * scale_cur * intermediate_guidance_multiplier * jac_action_cur

        if i < num_steps - 1:
            with torch.no_grad():
                x0_prime = net(x_euler, t_next, conditions, class_labels).to(torch.float64)
            d_prime = (x_euler - x0_prime) / t_next

            if use_obs:
                g_lin_prime, g_nonlin_prime, soft_resid_norm_prime, nll_prime = likelihood_stage(x0_prime, t_next)
                cskip_prime = _c_skip(t_next, edm_sigma_data, x_euler.device, x_euler.dtype)
                coutcin_prime = _c_out_c_in(t_next, edm_sigma_data, x_euler.device, x_euler.dtype)

                jf_lin_prime, jf_nonlin_prime = _local_weighted_fbar_jf_transpose_actions(
                    x_euler, x0_prime, [g_lin_prime, g_nonlin_prime], t_next, edm_sigma_data, idx_all,
                    radius=jf_local_radius, bandwidth_factor=jf_weight_bandwidth_factor,
                    ridge_rel=jf_ridge_rel, ridge_abs=jf_ridge_abs
                )

                direct_prime = g_lin_prime + g_nonlin_prime
                jf_prime = jf_scale_lin * jf_lin_prime + jf_scale_nonlin * jf_nonlin_prime
                jac_action_prime = cskip_prime * direct_prime + coutcin_prime * jf_prime

                scale_shape = [soft_resid_norm_prime.shape[0]] + [1] * (direct_prime.ndim - 1)
                scale_prime = adaptive_dt_factor / soft_resid_norm_prime.reshape(scale_shape)

            x_next = x_hat + dt * 0.5 * (d_cur + d_prime)
            if use_obs:
                x_next = x_next - dt * 0.5 * (
                    t_hat * scale_cur * intermediate_guidance_multiplier * jac_action_cur
                    + t_next * scale_prime * jac_action_prime
                )
                neg_log_likelihoods.append(float(0.5 * (nll_cur.mean() + nll_prime.mean()).cpu()))
        else:
            x_next = x_euler
            if use_obs:
                neg_log_likelihoods.append(float(nll_cur.mean().cpu()))

    if use_obs:
        return x_next, neg_log_likelihoods
    return x_next


#### Note on the ensemble-space Jacobian action

The helper `_weighted_fbar_jf_transpose_action` is algebraically equivalent to the full weighted ridge regression used in the low-dimensional formulation:

$$
B=(A^TA+\lambda I)^{-1}A^TC,
\qquad
B=J_F^T.
$$

Instead of forming $B$, the notebook evaluates

$$
Bg
=
A^T(AA^T+\lambda I)^{-1}Cg.
$$

This changes the computational implementation, not the regression model.


In [10]:
# check_obs_guidance_setup(norm_stats, obs_flat_idx, n_target, K, device="cpu")

## Phase 4: DA loop (re-runnable)

In [11]:
obs_sigma = 0.025

## 4a: Truth + DA

In [12]:
SAVE_RESULTS = False
SAVE_PATH = os.path.join(trajectory_dir, "da_ensf_dm_demo.pt")

core_kw = params.to_core_kwargs()
truth = Lorenz96Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth.set_state(state_after_burn)

In [13]:
n_burn = path_burn.shape[0]  # T_burn+1
burn_norm = [normalize_tensor(path_burn[i], norm_stats) for i in range(n_burn)]
truth_frames = [path_burn[i].clone() for i in range(n_burn)]  # physical (C, H, W)

In [14]:
hist_forecast = []

In [15]:
for k in range(NE):
    hf = [burn_norm[i].clone() for i in range(n_burn)]
    hf[-1] = perturb_state(hf[-1], ENSEMBLE_PERTURB_STD, SEED + 1000 + k)
    hist_forecast.append(hf)

In [16]:
rnd = StackedRandomGenerator(device, [SEED_DM + k for k in range(NE)])

In [17]:
rmse_truth_vs_forecast = []
spread_forecast = []

In [18]:
cycle_times = []
pbar = tqdm.tqdm(range(N_DA_CYCLES), desc="DA cycles", unit="cycle")

DA cycles:   0%|          | 0/300 [00:00<?, ?cycle/s]

In [19]:
for cycle in pbar:
    t_cycle_start = time.perf_counter()

    truth.step(timesteps_per_output)
    x_truth = frame_tensor(truth.fields()["x"].detach().cpu(), K)
    x_truth_norm = normalize_tensor(x_truth, norm_stats)
    cond_f = build_condition_from_history_batch(hist_forecast, HISTORY).to(device)
    if not (torch.isfinite(cond_f).all()):
        print(f"Stopping at cycle {cycle}: non-finite DM conditions")
        break

    lat_f = rnd.randn([NE, n_target, K, 1], device=device)

    with torch.no_grad():
        sample_f_norm = edm_sampler_obs(net, lat_f, cond_f, num_steps=NUM_STEPS).to(torch.float32)
    if not torch.isfinite(sample_f_norm).all():
        print(f"Stopping at cycle {cycle}: non-finite DM forecast sample")
        break

    next_forecast = sample_f_norm.cpu()

    if not (torch.isfinite(next_forecast).all()):
        print(f"Stopping at cycle {cycle}: non-finite ensemble state")
        break

    for k in range(NE):
        hist_forecast[k].append(next_forecast[k])
        
    rmse_f, spr_f = ensemble_mean_rmse(hist_forecast, x_truth_norm)
    if not all(np.isfinite([rmse_f, spr_f])):
        for k in range(NE):
            hist_forecast[k].pop()
        print(f"Stopping at cycle {cycle}: non-finite RMSE/spread metrics")
        break

    rmse_truth_vs_forecast.append(rmse_f)
    spread_forecast.append(spr_f)

    cycle_times.append(time.perf_counter() - t_cycle_start)
    avg_step = sum(cycle_times) / len(cycle_times)
    remaining = avg_step * (N_DA_CYCLES - cycle - 1)
    pbar.set_postfix(step=f"{avg_step:.2f}s", remaining=f"{remaining:.0f}s")

DA cycles: 100%|██████████| 300/300 [02:30<00:00,  2.00cycle/s, remaining=0s, step=0.50s]  


In [20]:
n_da_cycles_completed = len(rmse_truth_vs_forecast)
print(
    f"Completed {n_da_cycles_completed}/{N_DA_CYCLES} DA cycles. "
    f"forecast {rmse_truth_vs_forecast[-1] if rmse_truth_vs_forecast else float('nan'):.4f}"
)
if cycle_times:
    print(f"Avg time per cycle: {sum(cycle_times) / len(cycle_times):.2f}s")

Completed 300/300 DA cycles. forecast 0.2830
Avg time per cycle: 0.50s


In [21]:
truth_frames = [path_burn[i].clone() for i in range(path_burn.shape[0])]
truth_solver_log = Lorenz96Solver(path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw)
truth_solver_log.set_state(state_after_burn)
for _ in range(n_da_cycles_completed):
    truth_solver_log.step(timesteps_per_output)
    truth_frames.append(frame_tensor(truth_solver_log.fields()["x"].detach().cpu(), K))

path_truth_full = torch.stack(truth_frames, dim=0)
path_forecast_ens = torch.stack([torch.stack(h, dim=0) for h in hist_forecast], dim=0)

In [22]:
from scipy.io import savemat

# Move tensors to CPU numpy arrays
path_truth_np = path_truth_full.detach().cpu().numpy()

In [23]:
path_forecast_np = denormalize_tensor(path_forecast_ens, norm_stats)

In [24]:
rmse_forecast = torch.tensor(rmse_truth_vs_forecast, dtype = torch.float32)
spread_forecast = torch.tensor(spread_forecast, dtype = torch.float32)

In [25]:
rmse_forecast_np = rmse_forecast.detach().cpu().numpy()
spread_forecast_np = spread_forecast.detach().cpu().numpy()

In [26]:
import os

In [27]:
# folder1 = "../../output_Lorenz96/NonlinObs/Finalized/Arctan"
# os.makedirs(folder1, exist_ok=True)

# filename = (
#     f"Truth_and_Forecast_Ncycles{N_DA_CYCLES}"
#     f"_Tburnin{T_BURN_IN}"
#     f"_NE{NE}"
#     f"_Lorenz96_Nonlinobs_MixedArctan_v1.mat"
# )

# save_path = os.path.join(folder1, filename)

In [28]:
# savemat(save_path, {
#     "path_truth_full": path_truth_np,
#     "path_noda": path_forecast_np,
#     "rmse_noda": rmse_forecast_np,
#     "spread_noda": spread_forecast_np,
#     "t_burn_in": T_BURN_IN,
#     "Ncycle": N_DA_CYCLES,
#     "Nen": NE,
# })

## 4b: EDM-EnSF

### Unified observation configurations and localized weighted-likelihood parameters

The five observation methods share one common configuration dictionary, while each method dictionary contains only its structural exceptions and its current starting guidance values.

The four main likelihood/Jacobian scaling parameters are:

- `guidance_scale_lin`: $\lambda_{\mathrm{lin}}$, applied to the linear likelihood-gradient contribution.
- `guidance_scale_nonlin`: $\lambda_{\mathrm{nonlin}}$, applied to the nonlinear likelihood-gradient contribution.
- `jf_scale_lin`: $\alpha_{F,\mathrm{lin}}$, relative scale of the localized $J_F^T$ action generated by the linear contribution.
- `jf_scale_nonlin`: $\alpha_{F,\mathrm{nonlin}}$, relative scale of the localized $J_F^T$ action generated by the nonlinear contribution.

The local regression also uses:

- `jf_local_radius`: periodic half-width $R$; local dimension is $2R+1$.
- `residual_soften_alpha`: controls the floor in $\rho_e$.
- `jf_weight_bandwidth_factor`: controls the Gaussian regression bandwidth.
- `jf_ridge_rel`, `jf_ridge_abs`: stabilize each small local regression.
- `intermediate_guidance_multiplier`: optional multiplier for the predictor-stage likelihood correction.

With `NE=36`, the default `jf_local_radius=2` gives local dimension $5$, so the regression is no longer trying to infer a 1000-dimensional relation from 36 ensemble members.

All of these can be overridden by the parameter-sweep section without copying the full method configuration.


### Observation-index utilities for all five configurations

In [29]:
def make_l96_fixed_idx(K, fixed_percent=5.0, device="cuda", seed_fixed=1234):
    n_fixed = int(round(fixed_percent / 100.0 * K))

    g = torch.Generator(device="cpu")
    g.manual_seed(seed_fixed)

    idx_fixed = torch.randperm(K, generator=g)[:n_fixed]
    idx_fixed = torch.sort(idx_fixed)[0]

    return idx_fixed.to(device)


def make_l96_random_idx_excluding(
    K,
    random_percent=10.0,
    exclude_idx=None,
    device="cuda",
    seed_random=None,
):
    n_random = int(round(random_percent / 100.0 * K))

    all_idx = torch.arange(K, dtype=torch.long)

    if exclude_idx is not None and exclude_idx.numel() > 0:
        exclude_cpu = exclude_idx.detach().cpu()
        available_mask = torch.ones(K, dtype=torch.bool)
        available_mask[exclude_cpu] = False
        available_idx = all_idx[available_mask]
    else:
        available_idx = all_idx

    if seed_random is not None:
        g = torch.Generator(device="cpu")
        g.manual_seed(seed_random)
        perm = torch.randperm(len(available_idx), generator=g)
    else:
        perm = torch.randperm(len(available_idx))

    idx_random = available_idx[perm[:n_random]]
    idx_random = torch.sort(idx_random)[0]

    return idx_random.to(device)


def make_l96_random_idx_excluding_safe(
    K,
    random_percent=10.0,
    exclude_idx=None,
    device="cuda",
    seed_random=None,
):
    n_random = int(round(random_percent / 100.0 * K))

    all_idx = torch.arange(K, dtype=torch.long)

    if exclude_idx is not None and exclude_idx.numel() > 0:
        exclude_cpu = exclude_idx.detach().cpu().unique()
        available_mask = torch.ones(K, dtype=torch.bool)
        available_mask[exclude_cpu] = False
        available_idx = all_idx[available_mask]
    else:
        available_idx = all_idx

    if n_random > available_idx.numel():
        raise ValueError(
            f"Cannot sample {n_random} nonlinear indices from only "
            f"{available_idx.numel()} available indices. "
            f"Reduce coverage or shorten the nonoverlap memory."
        )

    if seed_random is not None:
        g = torch.Generator(device="cpu")
        g.manual_seed(seed_random)
        perm = torch.randperm(len(available_idx), generator=g)
    else:
        perm = torch.randperm(len(available_idx))

    idx_random = available_idx[perm[:n_random]]
    idx_random = torch.sort(idx_random)[0]

    return idx_random.to(device)


def make_l96_random_block_idx_periodic(
    K, L, M, exclude_idx=None,
    device="cuda", seed_random=None, max_tries=10000):
    """
    Select L disjoint periodic blocks, each with M consecutive indices.
    Total number of selected indices is L*M.
    """
    n_target = L * M

    if exclude_idx is not None and exclude_idx.numel() > 0:
        exclude_cpu = exclude_idx.detach().cpu().unique()
    else:
        exclude_cpu = torch.empty(0, dtype=torch.long)

    g = torch.Generator(device="cpu")
    if seed_random is not None:
        g.manual_seed(seed_random)

    selected = set()
    starts = []

    for _ in range(max_tries):
        if len(starts) == L:
            break

        start = int(torch.randint(0, K, (1,), generator=g).item())
        block = [int((start + j) % K) for j in range(M)]

        if len(exclude_cpu) > 0:
            if torch.isin(torch.tensor(block), exclude_cpu).any().item():
                continue

        if any(idx in selected for idx in block):
            continue

        starts.append(start)
        selected.update(block)

    if len(starts) < L:
        raise ValueError(
            f"Could only select {len(starts)} disjoint blocks out of requested L={L}. "
            f"Try smaller L or M."
        )

    idx = torch.tensor(sorted(selected), dtype=torch.long)
    assert idx.numel() == n_target

    return idx.to(device), starts


def make_l96_stencil_idx_periodic_safe(
    K,
    L,
    distance,
    exclude_idx=None,
    device="cuda",
    seed_random=None,
):
    n_target = 2 * L

    if seed_random is not None:
        g = torch.Generator(device="cpu")
        g.manual_seed(seed_random)
        center = int(torch.randint(0, K, (1,), generator=g).item())
    else:
        g = None
        center = int(torch.randint(0, K, (1,)).item())

    idx_left = [((center - k * distance) % K) for k in range(1, L + 1)]
    idx_right = [((center + k * distance) % K) for k in range(1, L + 1)]

    idx = torch.tensor(idx_left + idx_right, dtype=torch.long)
    idx = torch.unique(idx, sorted=True)

    available_mask = torch.ones(K, dtype=torch.bool)

    if exclude_idx is not None and exclude_idx.numel() > 0:
        exclude_cpu = exclude_idx.detach().cpu().unique()
        available_mask[exclude_cpu] = False

    idx = idx[available_mask[idx]]
    n_missing = n_target - idx.numel()

    if n_missing > 0:
        available_idx = torch.arange(K, dtype=torch.long)[available_mask]
        available_idx = available_idx[~torch.isin(available_idx, idx)]

        if n_missing > available_idx.numel():
            raise ValueError(
                f"Cannot sample {n_target} nonlinear indices. "
                f"Only {idx.numel() + available_idx.numel()} available."
            )

        if g is not None:
            perm = torch.randperm(available_idx.numel(), generator=g)
        else:
            perm = torch.randperm(available_idx.numel())

        fill_idx = available_idx[perm[:n_missing]]
        idx = torch.cat([idx, fill_idx], dim=0)

    idx = torch.sort(idx)[0]

    return idx.to(device), center


def sample_L_from_percent_range(K, percent_min, percent_max, rng):
    n_min = int(np.ceil(percent_min / 100.0 * K))
    n_max = int(np.floor(percent_max / 100.0 * K))

    # Need n_nonlin = 2L, so use even counts only
    L_min = int(np.ceil(n_min / 2))
    L_max = int(np.floor(n_max / 2))

    if L_min > L_max:
        raise ValueError(
            f"No valid L gives 2L in [{percent_min}, {percent_max}]% of K={K}."
        )

    L = int(rng.integers(L_min, L_max + 1))
    n_nonlin = 2 * L
    actual_percent = 100.0 * n_nonlin / K

    return L, n_nonlin, actual_percent

In [30]:
def observe_l96_mixed_combined(
    x_norm, idx_lin, idx_nonlin, obs_sigma_lin, obs_sigma_nonlin, phi,
    generator=None):
    x_flat = x_norm.reshape(-1)

    idx_lin = idx_lin.to(x_flat.device)
    idx_nonlin = idx_nonlin.to(x_flat.device)

    y_lin = x_flat[idx_lin].to(torch.float32)
    y_nonlin = phi(x_flat[idx_nonlin]).to(torch.float32)

    if obs_sigma_lin > 0 and y_lin.numel() > 0:
        y_lin = y_lin + obs_sigma_lin * torch.randn(
            y_lin.shape,
            device=y_lin.device,
            generator=generator,
        )

    if obs_sigma_nonlin > 0 and y_nonlin.numel() > 0:
        y_nonlin = y_nonlin + obs_sigma_nonlin * torch.randn(
            y_nonlin.shape,
            device=y_nonlin.device,
            generator=generator,
        )

    y_obs = torch.cat([y_lin, y_nonlin], dim=0)
    return y_obs


def scale_ensemble_spread(X, gamma):
    mean = X.mean(dim=0, keepdim=True)
    return mean + gamma * (X - mean)


def observation_rmse_mixed_arctan(
    x_norm, y_obs, idx_lin, idx_nonlin, phi):
    x_flat = x_norm.reshape(-1)

    idx_lin = idx_lin.to(x_flat.device)
    idx_nonlin = idx_nonlin.to(x_flat.device)

    y_lin_clean = x_flat[idx_lin]
    y_nonlin_clean = phi(x_flat[idx_nonlin])

    y_clean = torch.cat([y_lin_clean, y_nonlin_clean], dim=0).cpu()
    y_obs_cpu = y_obs.reshape(-1).cpu()

    return torch.sqrt(torch.mean((y_clean - y_obs_cpu) ** 2)).item()


def mixed_obs_misfit(x_norm, y_obs, idx_lin, idx_nonlin, phi_nonlin):
    x_flat = x_norm.reshape(-1)

    n_lin = idx_lin.numel()
    y_lin = y_obs[:n_lin]
    y_nonlin = y_obs[n_lin:]

    pred_lin = x_flat[idx_lin]
    pred_nonlin = phi_nonlin(x_flat[idx_nonlin])

    rmse_lin = torch.sqrt(torch.mean((pred_lin - y_lin.to(x_flat.device))**2)).item()
    rmse_nonlin = torch.sqrt(torch.mean((pred_nonlin - y_nonlin.to(x_flat.device))**2)).item()

    return rmse_lin, rmse_nonlin

### Unified DA runner

In [31]:
def run_da_configuration(config_name, use_denom=False, keep_history=True, config_override=None):
    """Run one of the five original observation configurations.

    Each call reinitializes the observation-index RNGs, observation-noise realization,
    EDM latent RNGs, analysis ensemble, and truth solver so that parameter candidates
    are compared under the same stochastic realization.
    """
    if config_name not in OBS_CONFIG_OPTIONS:
        raise ValueError(f"Unknown config {config_name!r}; choose from {OBS_CONFIG_OPTIONS}")

    cfg_obs = build_obs_config(config_name, config_override)
    obs_sigma_lin_run = float(cfg_obs.get("obs_sigma_lin", obs_sigma_lin))
    obs_sigma_nonlin_run = float(cfg_obs.get("obs_sigma_nonlin", obs_sigma_nonlin))

    rng_obs_percent_local = np.random.default_rng(SEED_OBS_PERCENT)
    rng_obs_local = np.random.default_rng(seed=SEED_OBS_INDEX)
    rnd_local = StackedRandomGenerator(device, [SEED_DM + k for k in range(NE)])

    idx_lin_fixed_local = make_l96_fixed_idx(K=K, fixed_percent=cfg_obs["obs_percent_fix"], device=device, seed_fixed=1234)

    hist_analysis_local = []
    for k in range(NE):
        ha = [burn_norm[i].clone() for i in range(n_burn)]
        ha[-1] = perturb_state(ha[-1], ENSEMBLE_PERTURB_STD, SEED + 1000 + k)
        hist_analysis_local.append(ha)

    rmse_truth_vs_analysis_local = []
    spread_analysis_local = []
    obs_rmse_local = []
    neg_log_likelihood_last_local = []

    truth_a_local = Lorenz96Solver(
        path_burn[-1][0, :, 0].to(device=device, dtype=dtype), **core_kw
    )
    truth_a_local.set_state(state_after_burn)

    prev_nonlin_list = []
    DA_cycle = N_DA_CYCLES // 3

    cycle_times_local = []
    pbar = tqdm.tqdm(
        range(N_DA_CYCLES), desc=f"DA cycles [{config_name}, denom={use_denom}]",
        unit="cycle",
    )

    for cycle in pbar:
        t_cycle_start = time.perf_counter()
        idx_lin = idx_lin_fixed_local
        guidance_scale_nonlin_cycle = cfg_obs.get("guidance_scale_nonlin", None)

        if config_name == "random":
            seed_obs_nonlin = int(rng_obs_local.integers(0, 2**32 - 1))
            obs_percent_random_DA = rng_obs_percent_local.uniform(
                cfg_obs["obs_min"], cfg_obs["obs_max"]
            )
            idx_nonlin = make_l96_random_idx_excluding(
                K=K, random_percent=obs_percent_random_DA, exclude_idx=idx_lin_fixed_local,
                device=device, seed_random=seed_obs_nonlin,
            )

        elif config_name == "timevarying":
            if cycle < 80:
                obs_min_cycle, obs_max_cycle = cfg_obs["obs_min_high"], cfg_obs["obs_max_high"]
                guidance_scale_nonlin_cycle = cfg_obs["guidance_scale_nonlin_high"]
            elif cycle < 2 * DA_cycle:
                obs_min_cycle, obs_max_cycle = cfg_obs["obs_min_mid"], cfg_obs["obs_max_mid"]
                guidance_scale_nonlin_cycle = cfg_obs["guidance_scale_nonlin_mid"]
            else:
                obs_min_cycle, obs_max_cycle = cfg_obs["obs_min"], cfg_obs["obs_max"]
                guidance_scale_nonlin_cycle = cfg_obs["guidance_scale_nonlin_low"]

            seed_obs_nonlin = int(rng_obs_local.integers(0, 2**32 - 1))
            obs_percent_random_DA = rng_obs_percent_local.uniform(obs_min_cycle, obs_max_cycle)
            idx_nonlin = make_l96_random_idx_excluding(
                K=K, random_percent=obs_percent_random_DA, exclude_idx=idx_lin_fixed_local,
                device=device, seed_random=seed_obs_nonlin,
            )

        elif config_name == "nooverlap":
            seed_obs_nonlin = int(rng_obs_local.integers(0, 2**32 - 1))
            obs_percent_random_DA = rng_obs_percent_local.uniform(
                cfg_obs["obs_min"], cfg_obs["obs_max"]
            )

            exclude_list = [idx_lin.detach().cpu()]
            for old_idx in prev_nonlin_list:
                exclude_list.append(old_idx.detach().cpu())
            exclude_idx = torch.cat(exclude_list).unique().to(device)

            idx_nonlin = make_l96_random_idx_excluding_safe(
                K=K, random_percent=obs_percent_random_DA, exclude_idx=exclude_idx,
                device=device, seed_random=seed_obs_nonlin,
            )

            prev_nonlin_list.append(idx_nonlin.detach().clone())
            if len(prev_nonlin_list) > cfg_obs["memory_steps"]:
                prev_nonlin_list.pop(0)

        elif config_name == "block":
            seed_obs_nonlin = int(rng_obs_local.integers(0, 2**32 - 1))
            idx_nonlin, block_starts = make_l96_random_block_idx_periodic(
                K=K, L=cfg_obs["L_block"], M=cfg_obs["M_block"], exclude_idx=idx_lin,
                device=device, seed_random=seed_obs_nonlin,
            )

        elif config_name == "stencil":
            seed_obs_nonlin = int(rng_obs_local.integers(0, 2**32 - 1))
            L_nonlin, n_nonlin, obs_percent_nonlin_actual = sample_L_from_percent_range(
                K=K, percent_min=cfg_obs["obs_min"], percent_max=cfg_obs["obs_max"],
                rng=rng_obs_percent_local,
            )
            if cfg_obs.get("distance_mode", "range") == "fixed":
                distance = int(cfg_obs["distance_fixed"])
            else:
                distance = int(rng_obs_local.integers(cfg_obs["distance_min"], cfg_obs["distance_max"] + 1))
            idx_nonlin, center_nonlin = make_l96_stencil_idx_periodic_safe(
                K=K, L=L_nonlin, distance=distance, exclude_idx=idx_lin, device=device, seed_random=seed_obs_nonlin
            )

        truth_a_local.step(timesteps_per_output)
        x_truth_a = frame_tensor(truth_a_local.fields()["x"].detach().cpu(), K)
        x_truth_norm_a = normalize_tensor(x_truth_a, norm_stats)

        obs_noise_gen = torch.Generator(device=x_truth_norm_a.device)
        obs_noise_gen.manual_seed(SEED_OBS_NOISE + cycle)
        y_obs = observe_l96_mixed_combined(
            x_truth_norm_a, idx_lin, idx_nonlin, obs_sigma_lin_run, obs_sigma_nonlin_run, phi=phi_nonlin, generator=obs_noise_gen
        ).to(device)

        if not torch.isfinite(y_obs).all():
            print(f"Stopping at cycle {cycle}: non-finite observations y_obs")
            break

        obs_rmse_local.append(
            observation_rmse_mixed_arctan(
                x_truth_norm_a, y_obs, idx_lin, idx_nonlin, phi_nonlin
            )
        )

        cond_a = build_condition_from_history_batch(hist_analysis_local, HISTORY).to(device)
        if not torch.isfinite(cond_a).all():
            print(f"Stopping at cycle {cycle}: non-finite DM conditions")
            break

        lat_a = cfg_obs["latent_inflation"] * rnd_local.randn([NE, n_target, K, 1], device=device)

        if (cycle + 1) % assim_gap == 0:
            obs_cfg = {
                "y_obs": y_obs,
                "idx_lin": idx_lin,
                "idx_nonlin": idx_nonlin,
                "obs_sigma_lin": obs_sigma_lin_run,
                "obs_sigma_nonlin": obs_sigma_nonlin_run,
                "phi_nonlin": phi_nonlin,
                "guidance_scale_lin": cfg_obs["guidance_scale_lin"],
                "guidance_scale_nonlin": guidance_scale_nonlin_cycle,
                "sigma_data": SIGMA_DATA,
                "obs_ell_lin": OBS_ELL_LIN,
                "obs_ell_nonlin": OBS_ELL_NONLIN,
                "use_denom": use_denom,
                "norm_stats": norm_stats,
                "jf_scale_lin": cfg_obs["jf_scale_lin"],
                "jf_scale_nonlin": cfg_obs["jf_scale_nonlin"],
                "jf_local_radius": cfg_obs["jf_local_radius"],
                "residual_soften_alpha": cfg_obs["residual_soften_alpha"],
                "jf_weight_bandwidth_factor": cfg_obs["jf_weight_bandwidth_factor"],
                "jf_ridge_rel": cfg_obs["jf_ridge_rel"],
                "jf_ridge_abs": cfg_obs["jf_ridge_abs"],
                "intermediate_guidance_multiplier": cfg_obs["intermediate_guidance_multiplier"],
                "edm_sigma_data_fallback": cfg_obs["edm_sigma_data_fallback"],
            }

            sample_a_norm, neg_log_likelihoods = edm_sampler_obs(net, lat_a, cond_a, obs=obs_cfg, num_steps=NUM_STEPS)
        else:
            sample_a_norm = edm_sampler_obs(net, lat_a, cond_a, obs=None, num_steps=NUM_STEPS)
            neg_log_likelihoods = []

        if not torch.isfinite(sample_a_norm).all():
            print(f"Stopping at cycle {cycle}: non-finite observation-guided DM sample")
            break

        sample_a_norm = scale_ensemble_spread(sample_a_norm, cfg_obs["posterior_gamma"])
        next_analysis = sample_a_norm.cpu().float()

        if not torch.isfinite(next_analysis).all():
            print(f"Stopping at cycle {cycle}: non-finite ensemble state")
            break

        for k in range(NE):
            hist_analysis_local[k].append(next_analysis[k])

        rmse_a, spr_a = ensemble_mean_rmse(hist_analysis_local, x_truth_norm_a)

        if not all(np.isfinite([rmse_a, spr_a])):
            for k in range(NE):
                hist_analysis_local[k].pop()
            print(f"Stopping at cycle {cycle}: non-finite RMSE/spread metrics")
            break

        neg_log_likelihood_last_local.append(
            neg_log_likelihoods[-1] if neg_log_likelihoods else float("nan")
        )
        rmse_truth_vs_analysis_local.append(rmse_a)
        spread_analysis_local.append(spr_a)

        cycle_times_local.append(time.perf_counter() - t_cycle_start)
        avg_step = sum(cycle_times_local) / len(cycle_times_local)
        remaining = avg_step * (N_DA_CYCLES - cycle - 1)
        pbar.set_postfix(step=f"{avg_step:.2f}s", remaining=f"{remaining:.0f}s")

    result = {
        "config_name": config_name,
        "use_denom": bool(use_denom),
        "config": cfg_obs,
        "idx_lin_fixed": idx_lin_fixed_local,
        "rmse_truth_vs_analysis": rmse_truth_vs_analysis_local,
        "spread_analysis": spread_analysis_local,
        "obs_rmse": obs_rmse_local,
        "neg_log_likelihood_last": neg_log_likelihood_last_local,
        "cycle_times": cycle_times_local,
    }
    if keep_history:
        result["hist_analysis"] = hist_analysis_local

    return result

In [ ]:
# Five available observation configurations:
#   1. "random"      : mixed fixed-linear + random nonlinear observations
#   2. "timevarying" : random nonlinear coverage decreases during the DA window
#   3. "nooverlap"   : random nonlinear observations avoid the previous memory_steps sets
#   4. "block"       : periodic random blocks
#   5. "stencil"     : periodic structured/stencil observations

OBS_CONFIG_OPTIONS = ("random", "timevarying", "nooverlap", "block", "stencil")
OBS_CONFIG = "random"
USE_OBS_DENOM = False

SEED_OBS_PERCENT = 111222
SEED_OBS_INDEX = 23456
SEED_OBS_NOISE = 34567

OBS_ALPHA = 12.0
phi_nonlin = lambda u: torch.atan(OBS_ALPHA * u)
assim_gap = 1

obs_sigma_lin = 0.02
obs_sigma_nonlin = 0.015
SIGMA_DATA = 0.32
OBS_ELL_LIN = 0.1
OBS_ELL_NONLIN = 0.1

# Parameters shared by the methods unless a method-specific exception or sweep override is supplied.
COMMON_OBS_CONFIG = {
    "obs_percent_fix": 4.0,
    "obs_min": 6.0,
    "obs_max": 8.0,
    "latent_inflation": 1.035,
    "posterior_gamma": 0.9775,
    "obs_sigma_lin": obs_sigma_lin,
    "obs_sigma_nonlin": obs_sigma_nonlin,

    # Localized weighted ensemble-Jacobian likelihood scaling.
    "jf_scale_lin": 1.0,
    "jf_scale_nonlin": 1.0,
    "jf_local_radius": 3,
    "residual_soften_alpha": 1.0,
    "jf_weight_bandwidth_factor": 1.0,
    "jf_ridge_rel": 1e-6,
    "jf_ridge_abs": 1e-10,
    "intermediate_guidance_multiplier": 1.0,
    "edm_sigma_data_fallback": 0.5,
}

# The current guidance values are retained only as starting points and should be re-tuned.
METHOD_CONFIGS = {
    "random": {
        "guidance_scale_lin": 0.001,
        "guidance_scale_nonlin": 0.000135,
        "jf_scale_lin": 0.0625,
        "jf_scale_nonlin": 0.125,
        "jf_local_radius": 4,
    },
    "timevarying": {
        "obs_min_high": 24.0,
        "obs_max_high": 26.0,
        "obs_min_mid": 16.0,
        "obs_max_mid": 18.0,
        "guidance_scale_lin": 0.0004,
        "guidance_scale_nonlin_high": 0.0000056,
        "guidance_scale_nonlin_mid": 0.0000056,
        "guidance_scale_nonlin_low": 0.00000515,
    },
    "nooverlap": {
        "memory_steps": 8,
        "guidance_scale_lin": 0.0006,
        "guidance_scale_nonlin": 0.000115,
        "jf_scale_lin": 0.0625,
        "jf_scale_nonlin": 0.125,
        "jf_local_radius": 4,
    },
    "block": {
        "L_block": 12,
        "M_block": 10,
        "guidance_scale_lin": 0.0009,
        "guidance_scale_nonlin": 0.0001,
        "jf_scale_lin": 0.0625,
        "jf_scale_nonlin": 0.205,
        "jf_local_radius": 4,
        "latent_inflation": 1.03,
        "posterior_gamma": 0.9875,
    },
    "stencil": {
        "distance_mode": "range",
        "distance_min": 5,
        "distance_max": 8,
        "distance_fixed": 10,
        "guidance_scale_lin": 0.0009,
        "guidance_scale_nonlin": 0.0001455,
        "jf_scale_lin": 0.0625,
        "jf_scale_nonlin": 0.125,
        "jf_local_radius": 4,
    },
}


def build_obs_config(config_name, override=None):
    if config_name not in OBS_CONFIG_OPTIONS:
        raise ValueError(f"config_name must be one of {OBS_CONFIG_OPTIONS}; got {config_name!r}")

    cfg = dict(COMMON_OBS_CONFIG)
    cfg.update(METHOD_CONFIGS[config_name])

    if config_name == "block":
        cfg["obs_percent_fix"] = 0.0

    if override is not None:
        override = dict(override)

        # Convenience: for timevarying, one nonlinear scale can be swept across all three windows.
        if config_name == "timevarying" and "guidance_scale_nonlin" in override:
            value = float(override["guidance_scale_nonlin"])
            override.setdefault("guidance_scale_nonlin_high", value)
            override.setdefault("guidance_scale_nonlin_mid", value)
            override.setdefault("guidance_scale_nonlin_low", value)

        cfg.update(override)

    if config_name == "stencil" and cfg["distance_mode"] not in ("range", "fixed"):
        raise ValueError("For stencil, distance_mode must be 'range' or 'fixed'.")

    return cfg


OBS_CONFIGS = {name: build_obs_config(name) for name in OBS_CONFIG_OPTIONS}

if OBS_CONFIG not in OBS_CONFIG_OPTIONS:
    raise ValueError(f"OBS_CONFIG must be one of {OBS_CONFIG_OPTIONS}; got {OBS_CONFIG!r}")

print(f"Selected configuration: {OBS_CONFIG}")
print(f"USE_OBS_DENOM = {USE_OBS_DENOM}")
print("Merged configuration:", OBS_CONFIGS[OBS_CONFIG])


Selected configuration: random
USE_OBS_DENOM = False
Merged configuration: {'obs_percent_fix': 4.0, 'obs_min': 6.0, 'obs_max': 8.0, 'latent_inflation': 1.035, 'posterior_gamma': 0.9775, 'obs_sigma_lin': 0.02, 'obs_sigma_nonlin': 0.015, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.125, 'jf_local_radius': 4, 'residual_soften_alpha': 1.0, 'jf_weight_bandwidth_factor': 1.0, 'jf_ridge_rel': 1e-06, 'jf_ridge_abs': 1e-10, 'intermediate_guidance_multiplier': 1.0, 'edm_sigma_data_fallback': 0.5, 'guidance_scale_lin': 0.001, 'guidance_scale_nonlin': 0.000135}


### Parameter sweeps for fine-tuning

Each candidate run starts from the same truth state, observation-index seeds, the same observation-noise realization at every DA cycle, and the same diffusion latent seeds. Therefore differences across a sweep are caused by the overridden parameters rather than different random realizations.

You only list the parameters that change. The remaining values are inherited automatically from the selected observation method.

A useful first-stage sweep for the localized weighted likelihood transport is

```python
SWEEP_PARAMETER_SETS = make_parameter_grid({
    "guidance_scale_lin": [0.5e-4, 1.0e-4, 2.0e-4, 4.0e-4],
    "guidance_scale_nonlin": [1.0e-6, 2.5e-6, 5.0e-6],
    "jf_scale_lin": [0.5, 1.0, 1.5],
    "jf_scale_nonlin": [0.5, 1.0, 1.5],
})
```

These numbers are examples, not recommended final values. The local transport is different from both the direct-guidance and global weighted-$F$ notebooks, so the parameter ranges should be refined after inspecting the first sweep.

You can then tune the stabilization/regression parameters separately, for example

```python
SWEEP_PARAMETER_SETS = make_parameter_grid({
    "jf_local_radius": [1, 2, 3],
    "residual_soften_alpha": [0.5, 1.0, 1.5],
    "jf_weight_bandwidth_factor": [0.75, 1.0, 1.25],
    "intermediate_guidance_multiplier": [0.5, 1.0, 1.5],
})
```

For `timevarying`, sweeping one `guidance_scale_nonlin` automatically applies that value to all three windows. If you want different values, sweep `guidance_scale_nonlin_high`, `guidance_scale_nonlin_mid`, and `guidance_scale_nonlin_low` explicitly.

After the sweep, the notebook plots the RMSE trajectories, stores the top-ranked override as `BEST_SWEEP_OVERRIDE`, and reruns that case with `keep_history=True` for analysis and saving.


In [33]:
from itertools import product


def make_parameter_grid(parameter_grid):
    keys = list(parameter_grid)
    return [dict(zip(keys, values)) for values in product(*(parameter_grid[key] for key in keys))]


def run_parameter_sweep(config_name, parameter_sets, use_denom=USE_OBS_DENOM):
    rows = []

    for run_id, override in enumerate(parameter_sets, start=1):
        print(f"\nSweep {run_id}/{len(parameter_sets)}: {config_name}, override={override}")
        result = run_da_configuration(config_name, use_denom=use_denom, keep_history=False, config_override=override)
        rmse = np.asarray(result["rmse_truth_vs_analysis"], dtype=np.float64)
        spread = np.asarray(result["spread_analysis"], dtype=np.float64)

        rows.append({
            "run_id": run_id,
            "config_name": config_name,
            "override": dict(override),
            "n_completed": len(rmse),
            "final_rmse": float(rmse[-1]) if len(rmse) else float("nan"),
            "mean_rmse": float(np.mean(rmse)) if len(rmse) else float("nan"),
            "final_spread": float(spread[-1]) if len(spread) else float("nan"),
            "mean_spread": float(np.mean(spread)) if len(spread) else float("nan"),
            "rmse_curve": rmse,
            "spread_curve": spread,
        })

    rows.sort(key=lambda row: (row["n_completed"] != N_DA_CYCLES, row["final_rmse"], row["mean_rmse"]))

    print("\nRanked sweep results:")
    for row in rows:
        print(f"run={row['run_id']:2d} completed={row['n_completed']:3d} final_RMSE={row['final_rmse']:.6f} mean_RMSE={row['mean_rmse']:.6f} override={row['override']}")

    return rows


In [52]:
# Fine-tuning example. Set RUN_PARAMETER_SWEEP = True when you want to run it.
RUN_PARAMETER_SWEEP = True
# SWEEP_CONFIG = "random"

SWEEP_CONFIG = "block"

# Small example: two candidate parameter sets without copying the full configuration.
SWEEP_PARAMETER_SETS = [
    {"guidance_scale_lin": 0.0009, "guidance_scale_nonlin": 0.0001, "jf_scale_lin": 0.0625, "jf_scale_nonlin": 0.205, "jf_local_radius": 4},
    {"guidance_scale_lin": 0.0009, "guidance_scale_nonlin": 0.00015, "jf_scale_lin": 0.0625, "jf_scale_nonlin": 0.205, "jf_local_radius": 4},
    {"guidance_scale_lin": 0.0009, "guidance_scale_nonlin": 0.0002, "jf_scale_lin": 0.0625, "jf_scale_nonlin": 0.205, "jf_local_radius": 4},
    {"guidance_scale_lin": 0.0009, "guidance_scale_nonlin": 0.00025, "jf_scale_lin": 0.0625, "jf_scale_nonlin": 0.205, "jf_local_radius": 4},
   
]

# For a Cartesian sweep, replace the list above by, e.g.
# SWEEP_PARAMETER_SETS = make_parameter_grid({
#     "guidance_scale_lin": [0.0002, 0.0003, 0.0004],
#     "guidance_scale_nonlin": [0.0000025, 0.0000050],
#     "jf_scale_lin": [0.5, 1.0, 1.5],
#     "jf_scale_nonlin": [0.5, 1.0, 1.5],
#     "jf_local_radius": [1, 2, 3],
# })

if RUN_PARAMETER_SWEEP:
    sweep_results = run_parameter_sweep(SWEEP_CONFIG, SWEEP_PARAMETER_SETS, use_denom=USE_OBS_DENOM)



Sweep 1/4: block, override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.0001, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}


DA cycles [block, denom=False]: 100%|██████████| 300/300 [02:53<00:00,  1.73cycle/s, remaining=0s, step=0.58s]  



Sweep 2/4: block, override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.00015, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}


DA cycles [block, denom=False]: 100%|██████████| 300/300 [02:54<00:00,  1.72cycle/s, remaining=0s, step=0.58s]  



Sweep 3/4: block, override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.0002, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}


DA cycles [block, denom=False]: 100%|██████████| 300/300 [02:56<00:00,  1.70cycle/s, remaining=0s, step=0.59s]  



Sweep 4/4: block, override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.00025, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}


DA cycles [block, denom=False]: 100%|██████████| 300/300 [02:57<00:00,  1.69cycle/s, remaining=0s, step=0.59s]  


Ranked sweep results:
run= 1 completed=300 final_RMSE=0.068659 mean_RMSE=0.058241 override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.0001, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}
run= 2 completed=300 final_RMSE=0.074177 mean_RMSE=0.063082 override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.00015, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}
run= 3 completed=300 final_RMSE=0.085077 mean_RMSE=0.071396 override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.0002, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}
run= 4 completed=300 final_RMSE=0.089666 mean_RMSE=0.076335 override={'guidance_scale_lin': 0.0009, 'guidance_scale_nonlin': 0.00025, 'jf_scale_lin': 0.0625, 'jf_scale_nonlin': 0.205, 'jf_local_radius': 4}


In [37]:
if RUN_PARAMETER_SWEEP:
    fig, ax = plt.subplots(figsize=(9, 5))

    for row in sweep_results:
        rmse = row["rmse_curve"]
        ax.plot(np.arange(1, len(rmse) + 1), rmse, label=f"run {row['run_id']}")

    ax.set_xlabel("DA cycle")
    ax.set_ylabel("RMSE")
    ax.set_title(f"{SWEEP_CONFIG}: parameter-sweep RMSE")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    best_sweep = sweep_results[0]
    BEST_SWEEP_OVERRIDE = best_sweep["override"]

    print("Best parameter set:")
    print(BEST_SWEEP_OVERRIDE)
    print(f"Final RMSE = {best_sweep['final_rmse']:.6f}")
    print(f"Mean RMSE  = {best_sweep['mean_rmse']:.6f}")


### Run the selected configuration

In [ ]:
if RUN_PARAMETER_SWEEP:
    selected_result = run_da_configuration(SWEEP_CONFIG, use_denom=USE_OBS_DENOM, keep_history=True, config_override=BEST_SWEEP_OVERRIDE)
    active_config_name = SWEEP_CONFIG
else:
    selected_result = run_da_configuration(OBS_CONFIG, use_denom=USE_OBS_DENOM, keep_history=True)
    active_config_name = OBS_CONFIG

# Unpack the selected run for the analysis/saving cells below.
hist_analysis = selected_result["hist_analysis"]
rmse_truth_vs_analysis = selected_result["rmse_truth_vs_analysis"]
spread_analysis = selected_result["spread_analysis"]
obs_rmse = selected_result["obs_rmse"]
neg_log_likelihood_last = selected_result["neg_log_likelihood_last"]
cycle_times = selected_result["cycle_times"]
idx_lin_fixed = selected_result["idx_lin_fixed"]

active_cfg = selected_result["config"]
obs_percent_fix = active_cfg["obs_percent_fix"]
GUIDANCE_SCALE_LIN = active_cfg["guidance_scale_lin"]
GUIDANCE_SCALE_NONLIN = active_cfg.get("guidance_scale_nonlin", float("nan"))
latent_inflation = active_cfg["latent_inflation"]
posterior_gamma = active_cfg["posterior_gamma"]
jf_scale_lin = active_cfg["jf_scale_lin"]
jf_scale_nonlin = active_cfg["jf_scale_nonlin"]
jf_local_radius = active_cfg["jf_local_radius"]
residual_soften_alpha = active_cfg["residual_soften_alpha"]
jf_weight_bandwidth_factor = active_cfg["jf_weight_bandwidth_factor"]
intermediate_guidance_multiplier = active_cfg["intermediate_guidance_multiplier"]
obs_sigma_lin_active = active_cfg["obs_sigma_lin"]
obs_sigma_nonlin_active = active_cfg["obs_sigma_nonlin"]

# Variables used by the filename/saving cells.
obs_min = active_cfg["obs_min"]
obs_max = active_cfg["obs_max"]
memory_steps = active_cfg.get("memory_steps", None)
L_block = active_cfg.get("L_block", None)
M_block = active_cfg.get("M_block", None)
distance_mode = active_cfg.get("distance_mode", None)
distance_min = active_cfg.get("distance_min", None)
distance_max = active_cfg.get("distance_max", None)
distance_fixed = active_cfg.get("distance_fixed", None)


In [49]:
import os
import json
import numpy as np
from scipy.io import savemat

In [50]:
def fmt_param(x):
    return f"{float(x):.10g}".replace(".", "p")


def fmt_sci(x):
    mant, exp = f"{float(x):.6e}".split("e")
    mant = mant.rstrip("0").rstrip(".").replace(".", "p")
    exp = int(exp)

    if exp < 0:
        exp_str = f"_{abs(exp):02d}"
    else:
        exp_str = f"p{exp:02d}"

    return f"{mant}e{exp_str}"

In [ ]:
# Convert analysis history and diagnostics to NumPy
# ------------------------------------------------------------------------------
path_analysis_ens = torch.stack(
    [torch.stack(h, dim=0) for h in hist_analysis],
    dim=0,
)

path_analysis_ens_phys = denormalize_tensor(
    path_analysis_ens,
    norm_stats,
)

path_analysis_np = (
    path_analysis_ens_phys
    .detach()
    .cpu()
    .numpy()
)

rmse_analysis_np = np.asarray(
    rmse_truth_vs_analysis,
    dtype=np.float32,
)

spread_analysis_np = np.asarray(
    spread_analysis,
    dtype=np.float32,
)

obs_rmse_np = np.asarray(
    obs_rmse,
    dtype=np.float32,
)

nll_np = np.asarray(
    neg_log_likelihood_last,
    dtype=np.float32,
)

cycle_times_np = np.asarray(
    cycle_times,
    dtype=np.float64,
)

n_completed = len(rmse_truth_vs_analysis)

In [ ]:
# Common filename information
# ------------------------------------------------------------------------------
denom_tag = "denom" if USE_OBS_DENOM else "nodenom"

common_name = (
    f"Lorenz96_DA_EnSF"
    f"_{active_config_name}"
    f"_{denom_tag}"
    f"_NE{NE}"
    f"_Ncycles{n_completed}"
    f"_gap{assim_gap}"
    f"_alpha{fmt_param(OBS_ALPHA)}"
    f"_sigLin{fmt_param(obs_sigma_lin_active)}"
    f"_sigNonlin{fmt_param(obs_sigma_nonlin_active)}"
)


# ------------------------------------------------------------------------------
# Configuration-specific filename information
# ------------------------------------------------------------------------------
if active_config_name == "random":

    config_name = (
        f"_fix{fmt_param(active_cfg['obs_percent_fix'])}pct"
        f"_obsMin{fmt_param(active_cfg['obs_min'])}"
        f"_obsMax{fmt_param(active_cfg['obs_max'])}"
        f"_gLin{fmt_sci(active_cfg['guidance_scale_lin'])}"
        f"_gNonlin{fmt_sci(active_cfg['guidance_scale_nonlin'])}"
    )


elif active_config_name == "timevarying":

    config_name = (
        f"_fix{fmt_param(active_cfg['obs_percent_fix'])}pct"
        f"_high{fmt_param(active_cfg['obs_min_high'])}"
        f"to{fmt_param(active_cfg['obs_max_high'])}"
        f"_mid{fmt_param(active_cfg['obs_min_mid'])}"
        f"to{fmt_param(active_cfg['obs_max_mid'])}"
        f"_low{fmt_param(active_cfg['obs_min'])}"
        f"to{fmt_param(active_cfg['obs_max'])}"
        f"_gLin{fmt_sci(active_cfg['guidance_scale_lin'])}"
        f"_gNonlinHigh{fmt_sci(active_cfg['guidance_scale_nonlin_high'])}"
        f"_gNonlinMid{fmt_sci(active_cfg['guidance_scale_nonlin_mid'])}"
        f"_gNonlinLow{fmt_sci(active_cfg['guidance_scale_nonlin_low'])}"
    )


elif active_config_name == "nooverlap":

    config_name = (
        f"_fix{fmt_param(active_cfg['obs_percent_fix'])}pct"
        f"_obsMin{fmt_param(active_cfg['obs_min'])}"
        f"_obsMax{fmt_param(active_cfg['obs_max'])}"
        f"_memory{active_cfg['memory_steps']}"
        f"_gLin{fmt_sci(active_cfg['guidance_scale_lin'])}"
        f"_gNonlin{fmt_sci(active_cfg['guidance_scale_nonlin'])}"
    )


elif active_config_name == "block":

    config_name = (
        f"_fix{fmt_param(active_cfg['obs_percent_fix'])}pct"
        f"_L{active_cfg['L_block']}"
        f"_M{active_cfg['M_block']}"
        f"_gLin{fmt_sci(active_cfg['guidance_scale_lin'])}"
        f"_gNonlin{fmt_sci(active_cfg['guidance_scale_nonlin'])}"
    )


elif active_config_name == "stencil":

    distance_name = (
        f"_d{active_cfg['distance_fixed']}" if active_cfg["distance_mode"] == "fixed"
        else f"_dMin{active_cfg['distance_min']}_dMax{active_cfg['distance_max']}"
    )
    config_name = (
        f"_fix{fmt_param(active_cfg['obs_percent_fix'])}pct"
        f"_obsMin{fmt_param(active_cfg['obs_min'])}"
        f"_obsMax{fmt_param(active_cfg['obs_max'])}"
        f"{distance_name}"
        f"_gLin{fmt_sci(active_cfg['guidance_scale_lin'])}"
        f"_gNonlin{fmt_sci(active_cfg['guidance_scale_nonlin'])}"
    )


else:
    raise ValueError(f"Unknown configuration: {active_config_name}")


# Parameters that are important for all five configurations
common_tuning_name = (
    f"_latInfl{fmt_param(latent_inflation)}"
    f"_postGamma{fmt_param(posterior_gamma)}"
    f"_jfLin{fmt_param(jf_scale_lin)}"
    f"_jfNonlin{fmt_param(jf_scale_nonlin)}"
    f"_jfR{int(jf_local_radius)}"
    f"_soft{fmt_param(residual_soften_alpha)}"
    f"_jfBW{fmt_param(jf_weight_bandwidth_factor)}"
)


In [ ]:
# case_name = common_name + config_name + common_tuning_name
# filename_save = f"{case_name}_analysis.mat"

# # ------------------------------------------------------------------------------
# # Save folder
# # ------------------------------------------------------------------------------
# save_folder = "../../output_Lorenz96/NonlinObs/Finalized/Arctan"
# os.makedirs(save_folder, exist_ok=True)

# save_path = os.path.join(save_folder, filename_save)


# # ------------------------------------------------------------------------------
# # Data + experiment metadata
# # ------------------------------------------------------------------------------
# save_dict = {
#     # ----------------------------------------------------------
#     # Main results
#     # ----------------------------------------------------------
#     "path_analysis": path_analysis_np,
#     "rmse_analysis": rmse_analysis_np,
#     "spread_analysis": spread_analysis_np,
#     "obs_rmse_all": obs_rmse_np,
#     "neg_log_likelihood_last": nll_np,
#     "cycle_times": cycle_times_np,

#     # ----------------------------------------------------------
#     # General experiment information
#     # ----------------------------------------------------------
#     "obs_config": active_config_name,
#     "use_obs_denom": int(USE_OBS_DENOM),

#     "Nen": NE,
#     "Ncycle_requested": N_DA_CYCLES,
#     "Ncycle_completed": n_completed,

#     "assim_gap": assim_gap,
#     "obs_alpha": OBS_ALPHA,

#     "obs_sigma_lin": obs_sigma_lin_active,
#     "obs_sigma_nonlin": obs_sigma_nonlin_active,

#     "sigma_data_obs_denom": SIGMA_DATA,
#     "obs_ell_lin": OBS_ELL_LIN,
#     "obs_ell_nonlin": OBS_ELL_NONLIN,

#     "seed_obs_percent": SEED_OBS_PERCENT,
#     "seed_obs_index": SEED_OBS_INDEX,
#     "seed_obs_noise": SEED_OBS_NOISE,

#     # ----------------------------------------------------------
#     # Common tuning parameters
#     # ----------------------------------------------------------
#     "latent_inflation": latent_inflation,
#     "posterior_gamma": posterior_gamma,
#     "jf_scale_lin": jf_scale_lin,
#     "jf_scale_nonlin": jf_scale_nonlin,
#     "jf_local_radius": jf_local_radius,
#     "residual_soften_alpha": residual_soften_alpha,
#     "jf_weight_bandwidth_factor": jf_weight_bandwidth_factor,
#     "intermediate_guidance_multiplier": intermediate_guidance_multiplier,

#     # Save the complete selected configuration as text too.
#     "active_config_json": json.dumps(active_cfg),
#     "used_parameter_sweep": int(RUN_PARAMETER_SWEEP),
# }


# # ------------------------------------------------------------------------------
# # Save every scalar field from the active observation configuration
# # separately as well.
# #
# # Examples:
# #   cfg_obs_percent_fix
# #   cfg_obs_min
# #   cfg_obs_max
# #   cfg_guidance_scale_lin
# #   cfg_guidance_scale_nonlin
# #   cfg_memory_steps
# #   cfg_L_block
# #   ...
# # ------------------------------------------------------------------------------
# for key, value in active_cfg.items():
#     if isinstance(value, (int, float, np.integer, np.floating)):
#         save_dict[f"cfg_{key}"] = value


# # Fixed linear observation locations, when applicable
# if idx_lin_fixed is not None:
#     if torch.is_tensor(idx_lin_fixed):
#         save_dict["idx_lin_fixed"] = idx_lin_fixed.detach().cpu().numpy()
#     else:
#         save_dict["idx_lin_fixed"] = np.asarray(idx_lin_fixed)


# # ------------------------------------------------------------------------------
# # Save
# # ------------------------------------------------------------------------------
# savemat(save_path, save_dict)

# print("Saved DA result:")
# print(save_path)

### Optional: run all five configurations

Run this cell instead of repeatedly changing `OBS_CONFIG`.  It keeps only summary metrics for each configuration, rather than five full analysis histories, to avoid unnecessary memory use.  The selected configuration above remains available for the unchanged plotting/saving cells below.

In [46]:
def fmt_param(x):
    return f"{x:.10g}".replace(".", "p")

In [47]:
def fmt_sci(x):
    mant, exp = f"{x:.6e}".split("e")
    mant = mant.rstrip("0").rstrip(".").replace(".", "p")

    exp = int(exp)

    if exp < 0:
        exp_str = f"_{abs(exp):02d}"
    else:
        exp_str = f"p{exp:02d}"

    return f"{mant}e{exp_str}"

### Run several observation methods with their default parameter sets

Use `CONFIGS_TO_RUN = OBS_CONFIG_OPTIONS` to run all five methods once with their merged default settings. To run only a subset, replace it with something such as

```python
CONFIGS_TO_RUN = ("random", "block", "stencil")
```

This section is for one default run per method. If you want multiple parameter sets for the **same** method, use the fine-tuning sweep section above instead of duplicating the full method dictionary.


In [51]:
from scipy.io import savemat
import os

CONFIGS_TO_RUN = OBS_CONFIG_OPTIONS
save_folder = "../../output_Lorenz96/NonlinObs/Finalized/Arctan/WithDenom"
os.makedirs(save_folder, exist_ok=True)

all_config_results = {}

for config_name in CONFIGS_TO_RUN:
    print("\n" + "=" * 80)
    print(f"Running {config_name!r} with USE_OBS_DENOM={USE_OBS_DENOM}")
    print("=" * 80)

    _result = run_da_configuration(config_name, use_denom=USE_OBS_DENOM, keep_history=True)
    cfg = _result["config"]
    hist_analysis_config = _result["hist_analysis"]

    path_analysis_ens = torch.stack([torch.stack(h, dim=0) for h in hist_analysis_config], dim=0)
    path_analysis_np = denormalize_tensor(path_analysis_ens, norm_stats).detach().cpu().numpy()
    rmse_analysis_np = np.asarray(_result["rmse_truth_vs_analysis"], dtype=np.float32)
    spread_analysis_np = np.asarray(_result["spread_analysis"], dtype=np.float32)
    obs_rmse_np = np.asarray(_result["obs_rmse"], dtype=np.float32)
    nll_np = np.asarray(_result["neg_log_likelihood_last"], dtype=np.float32)
    cycle_times_np = np.asarray(_result["cycle_times"], dtype=np.float32)

    denom_tag = "denom" if USE_OBS_DENOM else "nodenom"
    guidance_lin_str = fmt_sci(cfg["guidance_scale_lin"])

    if config_name == "timevarying":
        guidance_nonlin_values = [cfg["guidance_scale_nonlin_high"], cfg["guidance_scale_nonlin_mid"], cfg["guidance_scale_nonlin_low"]]
        guidance_name = f"_scalenonlinmin{fmt_sci(min(guidance_nonlin_values))}_scalenonlinmax{fmt_sci(max(guidance_nonlin_values))}"
    else:
        guidance_name = f"_scalenonlin{fmt_sci(cfg['guidance_scale_nonlin'])}"

    case_name = (
        f"Lorenz96_DA_EnSF_{config_name}_{denom_tag}_Ncycles{N_DA_CYCLES}"
        f"_scalelin{guidance_lin_str}{guidance_name}"
        f"_jfLin{fmt_param(cfg['jf_scale_lin'])}_jfNonlin{fmt_param(cfg['jf_scale_nonlin'])}"
        f"_jfR{int(cfg['jf_local_radius'])}_soft{fmt_param(cfg['residual_soften_alpha'])}"
        f"_jfBW{fmt_param(cfg['jf_weight_bandwidth_factor'])}"
    )
    filename_save = f"{case_name}_analysis.mat"

    save_dict = {
        "path_analysis": path_analysis_np,
        "rmse_analysis": rmse_analysis_np,
        "spread_analysis": spread_analysis_np,
        "obs_rmse_all": obs_rmse_np,
        "neg_log_likelihood_last": nll_np,
        "cycle_times": cycle_times_np,
        "active_config_json": json.dumps(cfg),
    }
    for key, value in cfg.items():
        if isinstance(value, (int, float, np.integer, np.floating)):
            save_dict[f"cfg_{key}"] = value

    savemat(os.path.join(save_folder, filename_save), save_dict)
    print(f"Saved: {filename_save}")

    all_config_results[config_name] = {
        "rmse_truth_vs_analysis": _result["rmse_truth_vs_analysis"],
        "spread_analysis": _result["spread_analysis"],
        "n_completed": len(_result["rmse_truth_vs_analysis"]),
        "last_rmse": _result["rmse_truth_vs_analysis"][-1] if _result["rmse_truth_vs_analysis"] else float("nan"),
        "filename": filename_save,
    }

    del hist_analysis_config, path_analysis_ens, path_analysis_np, rmse_analysis_np, spread_analysis_np, obs_rmse_np, nll_np, cycle_times_np, _result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nSummary:")
for config_name, result in all_config_results.items():
    print(f"{config_name:12s}: completed={result['n_completed']:3d}, last RMSE={result['last_rmse']:.6f}, saved={result['filename']}")



Running 'random' with USE_OBS_DENOM=False


DA cycles [random, denom=False]: 100%|██████████| 300/300 [02:57<00:00,  1.69cycle/s, remaining=0s, step=0.59s]  


Saved: Lorenz96_DA_EnSF_random_nodenom_Ncycles300_scalelin1e_03_scalenonlin1p35e_04_jfLin0p0625_jfNonlin0p125_jfR4_soft1_jfBW1_analysis.mat

Running 'timevarying' with USE_OBS_DENOM=False


DA cycles [timevarying, denom=False]: 100%|██████████| 300/300 [02:58<00:00,  1.68cycle/s, remaining=0s, step=0.59s]  


Saved: Lorenz96_DA_EnSF_timevarying_nodenom_Ncycles300_scalelin4e_04_scalenonlinmin5p15e_06_scalenonlinmax5p6e_06_jfLin1_jfNonlin1_jfR3_soft1_jfBW1_analysis.mat

Running 'nooverlap' with USE_OBS_DENOM=False


DA cycles [nooverlap, denom=False]: 100%|██████████| 300/300 [02:54<00:00,  1.72cycle/s, remaining=0s, step=0.58s]  


Saved: Lorenz96_DA_EnSF_nooverlap_nodenom_Ncycles300_scalelin6e_04_scalenonlin1p15e_04_jfLin0p0625_jfNonlin0p125_jfR4_soft1_jfBW1_analysis.mat

Running 'block' with USE_OBS_DENOM=False


DA cycles [block, denom=False]: 100%|██████████| 300/300 [02:53<00:00,  1.73cycle/s, remaining=0s, step=0.58s]  


Saved: Lorenz96_DA_EnSF_block_nodenom_Ncycles300_scalelin6e_04_scalenonlin9p45e_05_jfLin0p0625_jfNonlin0p205_jfR4_soft1_jfBW1_analysis.mat

Running 'stencil' with USE_OBS_DENOM=False


DA cycles [stencil, denom=False]: 100%|██████████| 300/300 [02:55<00:00,  1.71cycle/s, remaining=0s, step=0.58s]  


Saved: Lorenz96_DA_EnSF_stencil_nodenom_Ncycles300_scalelin9e_04_scalenonlin1p455e_04_jfLin0p0625_jfNonlin0p125_jfR4_soft1_jfBW1_analysis.mat

Summary:
random      : completed=300, last RMSE=0.072400, saved=Lorenz96_DA_EnSF_random_nodenom_Ncycles300_scalelin1e_03_scalenonlin1p35e_04_jfLin0p0625_jfNonlin0p125_jfR4_soft1_jfBW1_analysis.mat
timevarying : completed=300, last RMSE=0.363653, saved=Lorenz96_DA_EnSF_timevarying_nodenom_Ncycles300_scalelin4e_04_scalenonlinmin5p15e_06_scalenonlinmax5p6e_06_jfLin1_jfNonlin1_jfR3_soft1_jfBW1_analysis.mat
nooverlap   : completed=300, last RMSE=0.058843, saved=Lorenz96_DA_EnSF_nooverlap_nodenom_Ncycles300_scalelin6e_04_scalenonlin1p15e_04_jfLin0p0625_jfNonlin0p125_jfR4_soft1_jfBW1_analysis.mat
block       : completed=300, last RMSE=0.070661, saved=Lorenz96_DA_EnSF_block_nodenom_Ncycles300_scalelin6e_04_scalenonlin9p45e_05_jfLin0p0625_jfNonlin0p205_jfR4_soft1_jfBW1_analysis.mat
stencil     : completed=300, last RMSE=0.084431, saved=Lorenz96_DA_EnSF_s

## Phase 5: Metrics vs assimilation cycle

In [ ]:
n_done = len(rmse_truth_vs_analysis)
cycles = np.arange(n_done)
fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
if n_done == 0:
    print("No completed DA cycles to plot.")
else:
    axs[0].plot(cycles, rmse_truth_vs_forecast, "o-", label="DM forecast (ens. mean)")
    axs[0].plot(cycles, rmse_truth_vs_analysis, "o-", label="DM + obs guidance (ens. mean)")
    axs[0].set_ylabel("RMSE vs truth")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)
    axs[1].plot(cycles, spread_forecast, "o-", label="forecast spread")
    axs[1].plot(cycles, spread_analysis, "o-", label="analysis spread")
    axs[1].set_ylabel("ens. spread (mean std)")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)
    axs[2].plot(cycles, obs_rmse[:n_done], "o-", color="gray", label="obs error")
    axs[2].set_ylabel("obs RMSE")
    axs[2].set_xlabel("assimilation cycle (after burn-in)")
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)
    plt.suptitle(f"L96 DA with NE={NE}: {n_done} cycles completed")
    plt.tight_layout()
    plt.show()


## Ring snapshots: truth vs ensemble-mean forecast / analysis

In [ ]:
t0 = path_burn.shape[0] - 1
truth_seg = to_seq(path_truth_full[t0:])
analysis_seg = to_seq(path_analysis[t0:])
forecast_seg = to_seq(path_forecast[t0:])
k_show = min(10, truth_seg.shape[0] - 1)
x_axis = np.arange(K)

fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axs[0].plot(x_axis, truth_seg[k_show].numpy(), color="C0")
axs[0].set_title(f"truth (L96 solver), k={k_show}")
axs[1].plot(x_axis, analysis_seg[k_show].numpy(), color="C1")
axs[1].set_title(f"DM + obs guidance analysis (ens. mean), k={k_show}")
axs[2].plot(x_axis, forecast_seg[k_show].numpy(), color="C2")
axs[2].set_title(f"DM forecast (ens. mean), k={k_show}")
for ax in axs:
    ax.set_ylabel("x")
    ax.grid(True, alpha=0.3)
axs[-1].set_xlabel("dimension index i")
plt.tight_layout()
plt.show()


## Animate: selected ensemble members

Saves animation to HTML file instead of embedding huge output in the notebook (prevents file corruption).


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

ANIM_ENS_MEMBERS = [0, 1, 2]
ANIM_MAX_FRAMES = 80
ANIM_OUT = os.path.join(trajectory_dir, "da_ensf_ring_compare.html")

t0 = path_burn.shape[0] - 1
truth_seg = to_seq(path_truth_full[t0:])
T_seg = min(
    truth_seg.shape[0],
    path_analysis_ens.shape[1] - t0,
    path_forecast_ens.shape[1] - t0,
    ANIM_MAX_FRAMES,
)
n_rows = len(ANIM_ENS_MEMBERS)
x_axis = np.arange(K)

stacked = [truth_seg[:T_seg]]
for k in ANIM_ENS_MEMBERS:
    stacked.append(to_seq(path_analysis_ens[k, t0 : t0 + T_seg]))
    stacked.append(to_seq(path_forecast_ens[k, t0 : t0 + T_seg]))
stacked = torch.stack(stacked, dim=0)
vmin, vmax = float(stacked.min()), float(stacked.max())

fig, axs = plt.subplots(n_rows, 3, figsize=(10, 2.8 * n_rows), squeeze=False)
fig.subplots_adjust(hspace=0.35, wspace=0.12)
ims = []
for r, k in enumerate(ANIM_ENS_MEMBERS):
    row_data = [
        truth_seg[:T_seg],
        to_seq(path_analysis_ens[k, t0 : t0 + T_seg]),
        to_seq(path_forecast_ens[k, t0 : t0 + T_seg]),
    ]
    col_labels = ["truth", f"analysis ens {k}", f"forecast ens {k}"]
    for c in range(3):
        ax = axs[r, c]
        (line,) = ax.plot(x_axis, row_data[c][0].numpy(), linewidth=1.0)
        ims.append((row_data[c], line))
        ax.set_ylim(vmin, vmax)
        ax.set_title(col_labels[c])
        ax.grid(True, alpha=0.3)
        if r == n_rows - 1:
            ax.set_xlabel("dimension i")
title = fig.suptitle("")


def update(frame_k):
    for row_data, line in ims:
        line.set_ydata(row_data[frame_k].numpy())
    title.set_text(f"post-burn-in k={frame_k}  |  cols: truth / analysis / forecast")
    return [line for _, line in ims] + [title]


anim = FuncAnimation(fig, update, frames=T_seg, interval=120, blit=True, repeat=True)
anim.save(ANIM_OUT, writer="html")
plt.close(fig)
print("Saved animation:", ANIM_OUT)
display(HTML(f'<a href="{ANIM_OUT}" target="_blank">Open ring comparison animation</a>'))
